# Master Thesis -- Longformer Pretraining (Phases 1-3)

Stateful Sequence Detection of Multi-Step BOLA -- pretraining pipeline.
Three-phase context-length curriculum (512 -> 2048 -> 8192 tokens). Phase 4 (32768) was
evaluated and dropped -- see the length-distribution check cell near the end and
`supervisor_notes/note_2026-07-12_phase4-curriculum-drop.md`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/thesis'

!pip install transformers datasets accelerate torch -q

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
import json

data_file = f"{BASE}/data/tokenised/benign_windows.jsonl"

sequences = []
with open(data_file) as f:
    for line in f:
        item = json.loads(line)
        sequences.append(item['sequence'])

print(f"Loaded {len(sequences)} training sequences")
print(f"\nSample sequence preview:")
print(sequences[0][:400])

In [ ]:
from transformers import LongformerTokenizer
from torch.utils.data import Dataset
import torch

tokenizer = LongformerTokenizer.from_pretrained('allenai/longformer-base-4096')

special_tokens = {
    'additional_special_tokens': [
        'METHOD:GET', 'METHOD:POST', 'METHOD:PUT', 'METHOD:DELETE', 'METHOD:PATCH',
        'ROLE:Admin', 'ROLE:User', 'ROLE:Anonymous', 'ROLE:Staff', 'ROLE:Manager',
        'ROLE:Customer', 'ROLE:Employee', 'ROLE:Uploader', 'ROLE:Client',
        'ROLE:LeaveManager', 'ROLE:Salary', 'ROLE:BorrowManager',
        'ROLE:Logistician', 'ROLE:Translator', 'ROLE:Salesman',
        'PARAMS:', 'PATH:', '[SEP]', '{id}', '{uuid}', '{hex}', '{val}'
    ]
}
tokenizer.add_special_tokens(special_tokens)
print(f"Vocabulary size: {len(tokenizer)}")

MAX_LENGTH = 512

class RestTraceDataset(Dataset):
    def __init__(self, sequences, tokenizer, max_length):
        self.encodings = tokenizer(
            sequences,
            truncation=True,
            max_length=max_length,
            padding='max_length',
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx]
        }

dataset = RestTraceDataset(sequences, tokenizer, MAX_LENGTH)
print(f"Dataset size: {len(dataset)} samples")
print(f"Input shape: {dataset[0]['input_ids'].shape}")

## Phase 1 -- 512 tokens

In [ ]:
from transformers import (
    LongformerForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
import os

CHECKPOINT_DIR = f"{BASE}/checkpoints/phase1"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model = LongformerForMaskedLM.from_pretrained('allenai/longformer-base-4096')
model.resize_token_embeddings(len(tokenizer))

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    save_steps=50,
    save_total_limit=3,
    logging_steps=10,
    fp16=True,
    dataloader_num_workers=0,
    report_to="none",
    prediction_loss_only=True,
)

trainer = Trainer(
    model=model, args=training_args,
    data_collator=data_collator, train_dataset=dataset,
)

checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('checkpoint')]
last_checkpoint = os.path.join(CHECKPOINT_DIR, sorted(checkpoints)[-1]) if checkpoints else None
print(f"Resuming from {last_checkpoint}" if last_checkpoint else "Starting fresh Phase 1")

trainer.train(resume_from_checkpoint=last_checkpoint)

model.save_pretrained(f"{BASE}/models/longformer/phase1_final")
tokenizer.save_pretrained(f"{BASE}/models/longformer/phase1_final")
print("Phase 1 complete!")

## Phase 2 -- 2048 tokens

In [ ]:
import os
from transformers import (
    LongformerForMaskedLM, LongformerTokenizer,
    DataCollatorForLanguageModeling, TrainingArguments, Trainer
)

CHECKPOINT_DIR_P2 = f"{BASE}/checkpoints/phase2"
os.makedirs(CHECKPOINT_DIR_P2, exist_ok=True)

MAX_LENGTH_P2 = 2048

model_p2 = LongformerForMaskedLM.from_pretrained(f"{BASE}/models/longformer/phase1_final")
tokenizer_p2 = LongformerTokenizer.from_pretrained(f"{BASE}/models/longformer/phase1_final")

dataset_p2 = RestTraceDataset(sequences, tokenizer_p2, MAX_LENGTH_P2)
print(f"Dataset size: {len(dataset_p2)} samples")

data_collator_p2 = DataCollatorForLanguageModeling(
    tokenizer=tokenizer_p2, mlm=True, mlm_probability=0.15
)

training_args_p2 = TrainingArguments(
    output_dir=CHECKPOINT_DIR_P2,
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    save_steps=50,
    save_total_limit=3,
    logging_steps=10,
    fp16=True,
    dataloader_num_workers=0,
    report_to="none",
    prediction_loss_only=True,
)

trainer_p2 = Trainer(
    model=model_p2, args=training_args_p2,
    data_collator=data_collator_p2, train_dataset=dataset_p2,
)

checkpoints = [d for d in os.listdir(CHECKPOINT_DIR_P2) if d.startswith('checkpoint')]
last_checkpoint = os.path.join(CHECKPOINT_DIR_P2, sorted(checkpoints)[-1]) if checkpoints else None
print(f"Resuming from {last_checkpoint}" if last_checkpoint else "Starting fresh Phase 2")

trainer_p2.train(resume_from_checkpoint=last_checkpoint)

model_p2.save_pretrained(f"{BASE}/models/longformer/phase2_final")
tokenizer_p2.save_pretrained(f"{BASE}/models/longformer/phase2_final")
print("Phase 2 complete!")

## Phase 3 -- 8192 tokens, WITH position-embedding extension fix

Longformer's native position embeddings only go to ~4098. Extending to 8192 requires
manually tiling the position-embedding weights (Beltagy et al.'s technique -- Brozik
et al. do not specify their own extension mechanism, so this cites the original
Longformer paper's method instead).

In [ ]:
import os
from transformers import (
    LongformerForMaskedLM, LongformerTokenizer,
    DataCollatorForLanguageModeling, TrainingArguments, Trainer
)

CHECKPOINT_DIR_P3 = f"{BASE}/checkpoints/phase3"
os.makedirs(CHECKPOINT_DIR_P3, exist_ok=True)

MAX_LENGTH_P3 = 8192

model_p3 = LongformerForMaskedLM.from_pretrained(f"{BASE}/models/longformer/phase2_final")
tokenizer_p3 = LongformerTokenizer.from_pretrained(f"{BASE}/models/longformer/phase2_final")
print("Loaded Phase 2 model")
print(f"Position embeddings BEFORE fix: {model_p3.config.max_position_embeddings}")

def extend_position_embeddings(model, new_max_pos):
    old_pos_embed = model.longformer.embeddings.position_embeddings.weight
    old_max_pos, embed_dim = old_pos_embed.shape
    k = 2  # RoBERTa/Longformer padding offset

    new_pos_embed = old_pos_embed.new_empty(new_max_pos, embed_dim)
    new_pos_embed[:k] = old_pos_embed[:k]

    step = old_max_pos - k
    pos = k
    while pos < new_max_pos:
        chunk = min(step, new_max_pos - pos)
        new_pos_embed[pos:pos+chunk] = old_pos_embed[k:k+chunk]
        pos += chunk

    model.longformer.embeddings.position_embeddings = torch.nn.Embedding.from_pretrained(new_pos_embed)
    model.config.max_position_embeddings = new_max_pos
    return model

model_p3 = extend_position_embeddings(model_p3, MAX_LENGTH_P3)
print(f"Position embeddings AFTER fix: {model_p3.config.max_position_embeddings}")

dataset_p3 = RestTraceDataset(sequences, tokenizer_p3, MAX_LENGTH_P3)
print(f"Dataset size: {len(dataset_p3)} samples")

data_collator_p3 = DataCollatorForLanguageModeling(
    tokenizer=tokenizer_p3, mlm=True, mlm_probability=0.15
)

training_args_p3 = TrainingArguments(
    output_dir=CHECKPOINT_DIR_P3,
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    save_steps=50,
    save_total_limit=3,
    logging_steps=10,
    fp16=True,
    dataloader_num_workers=0,
    report_to="none",
    prediction_loss_only=True,
    gradient_checkpointing=True,
)


trainer_p3 = Trainer(
    model=model_p3, args=training_args_p3,
    data_collator=data_collator_p3, train_dataset=dataset_p3,
)

checkpoints = [d for d in os.listdir(CHECKPOINT_DIR_P3) if d.startswith('checkpoint')]
last_checkpoint = os.path.join(CHECKPOINT_DIR_P3, sorted(checkpoints)[-1]) if checkpoints else None
print(f"Resuming from {last_checkpoint}" if last_checkpoint else "Starting fresh Phase 3")

trainer_p3.train(resume_from_checkpoint=last_checkpoint)

model_p3.save_pretrained(f"{BASE}/models/longformer/phase3_final")
tokenizer_p3.save_pretrained(f"{BASE}/models/longformer/phase3_final")
print("Phase 3 complete!")

## Corpus length-distribution check -> Phase 4 decision

Brozik et al.'s 32768-token Phase 4 target was derived from *their* corpus's 95th
percentile (raw Windows API traces). Checking the same statistic on our own corpus
before blindly copying their number.

In [ ]:
from transformers import LongformerTokenizer
import json, numpy as np

tokenizer = LongformerTokenizer.from_pretrained(f"{BASE}/models/longformer/phase3_final")
print(f"Vocab size: {len(tokenizer)}")

lengths = [len(tokenizer(seq)['input_ids']) for seq in sequences]
print(f"Min: {min(lengths)}, Max: {max(lengths)}, Mean: {np.mean(lengths):.0f}")
print(f"50th percentile: {np.percentile(lengths, 50):.0f}")
print(f"95th percentile: {np.percentile(lengths, 95):.0f}")
print(f"99th percentile: {np.percentile(lengths, 99):.0f}")

# DECISION: max=5026, 99th percentile=2564 -- both well under Phase 3's 8192 ceiling.
# Phase 4 (32768) would train almost entirely on padding. DROPPED.
# See supervisor_notes/note_2026-07-12_phase4-curriculum-drop.md

## Role-vocabulary check (Track A prep)

Checks which ROLE: values actually appear in the benign corpus per app, so that
hand-built Track A attack exemplars use identical role tokens to what the
pretrained model has already seen.

In [ ]:
import json

apps_seen = {}
with open(f"{BASE}/data/tokenised/benign_windows.jsonl") as f:
    for line in f:
        r = json.loads(line)
        cve = r.get('cve', 'UNKNOWN')
        role = r.get('role', 'UNKNOWN')
        apps_seen.setdefault(cve, set()).add(role)

print(f"Total distinct CVE/app tags: {len(apps_seen)}\n")
for cve, roles in sorted(apps_seen.items()):
    print(f"{cve:30s} -> {sorted(roles)}")